# DINO: test de las tres proyecciones entrenadas
## Fase 1. Preparación y verificación de datos
Objetivo: comprobar si la mejora de validación de D1 se mantiene en test. Pregunta: ¿la proyección mejora DINO congelado? Hipótesis: mejora posible, no garantizada. Evaluación intermedia tras desconexión de Colab; LoRA y combinación todavía pendientes. Se fijan antes del test las tres proyecciones publicadas y la semilla 46, seleccionada solo por validación. El protocolo pendiente de LoRA no se ajustará con estos resultados. Datos preparados: público v1 inmutable (277 fotos) y PetTrace v6 (502), galerías separadas. Sin limpieza ni entrenamiento aquí.

In [1]:
import csv, hashlib, importlib.metadata, json, os, platform, shutil, sys, tarfile, time
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
from uuid import uuid4
from dotenv import load_dotenv
import numpy as np
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # CPU interpolation for unsupported MPS operations.
import torch
import mlflow
from huggingface_hub import hf_hub_download, snapshot_download
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.benchmark import Dataset, Photo
from pettrace.data.refined_sources import RELEASES
from pettrace.data.verification import sha256
from pettrace.experiments.e1 import _private_dataset
from pettrace.experiments.zero_shot import save_publication_input, load_publication_input

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Python 3.11–3.13 required")
if not torch.backends.mps.is_available():
    raise RuntimeError("This fixed execution requires the Mac MPS device")
ROOT = Path.cwd()
assert (ROOT / "src/pettrace").is_dir(), "Run from the repository root"
load_dotenv(ROOT / ".env")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
os.environ["MLFLOW_REGISTRY_URI"] = PRODUCTION_URI
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if not os.environ.get(name):
        raise RuntimeError(f"Missing {name}")
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
PUBLIC_RUN = "358cb901f3e34483b24acc6ce93649bb"
MODEL_REPO = "facebook/dinov2-large"
MODEL_REVISION = "47b73eefe95e8d44ec3623f8890bd894b6ea2d6c"
LOG_TO_MLFLOW = True
RUN_LABEL = "20261118-projection-test-v1"  # Fixed study receipt; reruns resume publication.
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
REPORT = ROOT / "reports/dino-projection-test" / RUN_LABEL
REPORT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "reports/c1-dinov2-reference/cache"
CACHE.mkdir(exist_ok=True)
SOURCE = source_identity()
method_cells = json.loads((ROOT / "notebooks/dino_projection_test.ipynb").read_text())["cells"]
METHOD_SHA256 = hashlib.sha256(json.dumps([c["source"] for c in method_cells],sort_keys=True).encode()).hexdigest()
print("Study:", RUN_LABEL, "| Device: MPS | Package:", SOURCE["package_version"])

# Lock all recovered checkpoints and the pending study before observing test results.
recovered=json.loads((ROOT/'reports/dino-pipeline-session/recovered/verified.json').read_text())
assert sorted(r['seed'] for r in recovered)==[45,46,47]
selection=max(recovered,key=lambda r:(r['metrics']['val.mAP'],-r['seed']))['seed']
assert selection==46
locked=dict(selected_projection_seed=selection,training_runs=recovered,
 pending_study_sha256=sha256(ROOT/'notebooks/dino_pipeline.ipynb'),
 scope='Interim projection evaluation; full-family selection remains pending. No changes to LoRA from test outcomes.',
 registration='Deferred to the full study; REGISTER_MODEL=True there remains unchanged')
lock=REPORT/'locked-before-test.json'
if lock.exists():assert json.loads(lock.read_text())==locked
else:lock.write_text(json.dumps(locked,indent=2))

client = mlflow.MlflowClient()
receipt_path = Path(client.download_artifacts(PUBLIC_RUN, "report/dataset-selection.json", str(CACHE)))
receipt = json.loads(receipt_path.read_text())
assert receipt["source"]["commit"] == "a724c1099a5432609c853958e88b68c4cf82d871"
expected = {(r["animal"], r["species"], r["sha256"]) for r in receipt["rows"]}
photos, rows = [], []
for source in ("lcw", "dogs-world"):
    spec = RELEASES[source]
    root = Path(snapshot_download(spec["repo_id"], repo_type="dataset", revision=spec["v2_commit"],
        allow_patterns=["versions/v1/dataset.json", "versions/v1/metadata.csv", "versions/v1/data/test/*"]))
    manifest = json.loads((root / "versions/v1/dataset.json").read_text())
    meta = root / "versions/v1/metadata.csv"
    assert sha256(meta) == manifest["file_sha256"]["versions/v1/metadata.csv"]
    with meta.open(newline="") as f:
        all_rows = list(csv.DictReader(f))
    subset = [r for r in all_rows if r["split"] == "test"]
    assert not {r["identity_id"] for r in subset} & {r["identity_id"] for r in all_rows if r["split"] != "test"}
    by_hash = {r["sha256"]: r for r in subset}
    assert len(by_hash) == len(subset)
    target_dir = CACHE / "images" / source
    target_dir.mkdir(parents=True, exist_ok=True)
    found = set()
    for shard in manifest["shards"]:
        if "/data/test/" not in shard["path"]:
            continue
        assert not Path(shard["path"]).is_absolute() and ".." not in Path(shard["path"]).parts
        archive = root / shard["path"]
        assert sha256(archive) == shard["sha256"]
        with tarfile.open(archive) as tar:
            for member in tar:
                name = Path(member.name)
                assert member.isfile() and name.name == member.name
                if name.suffix == ".json":
                    continue
                assert name.suffix.lower() in {".jpg", ".jpeg", ".png"}
                assert name.stem in by_hash and name.stem not in found
                target = target_dir / name.name
                if not target.exists() or sha256(target) != name.stem:
                    data = tar.extractfile(member).read()
                    assert hashlib.sha256(data).hexdigest() == name.stem
                    target.write_bytes(data)
                row = by_hash[name.stem]
                photos.append(Photo(str(target), row["identity_id"], row["species"], "test", 0.0))
                rows.append((row["identity_id"], row["species"], name.stem))
                found.add(name.stem)
    assert found == set(by_hash)
assert set(rows) == expected and len(rows) == len(expected) == 277
identity = {"prepared_fingerprint": receipt["prepared_fingerprint"], "split": "test", "rows": sorted(rows)}
digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert digest == receipt["evaluation_fingerprint"]
public_ds = Dataset(receipt["source"]["repo"], "v2", receipt["source"]["commit"], digest, str(CACHE),
    {p.animal: {"species":p.species,"split":"test"} for p in photos}, photos)
private_ds = _private_dataset("v6", DATASET_COMMIT, CACHE / "private")
assert len(private_ds.photos) == 502 and private_ds.fingerprint == "6dcae9a47764beb0"
assert not {r[2] for r in rows} & {sha256(p.path) for p in private_ds.photos}
(REPORT / "dataset-selection.json").write_text(json.dumps(receipt, indent=2))
print("Verified:", len(public_ds.photos), "public photos /", len(public_ds.animals), "identities;",
      len(private_ds.photos), "private photos /", len(private_ds.animals), "identities")


2026/11/18 17:42:05 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Study: 20261118-projection-test-v1 | Device: MPS | Package: 0.2.42


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 506 files:   0%|          | 0/506 [00:00<?, ?it/s]

Verified: 277 public photos / 60 identities; 502 private photos / 50 identities


## Fase 2. Representación y entrenamiento
Se cargan las tres proyecciones entrenadas sobre 193.276 fotos train disponibles, selección por 273 val, semillas 45-47. No se reentrena. Control: DINOv2-L/14 fijado, mismo detector E2, procesador y normalización; intervención: matriz 1024x1024 entrenada con batch-hard triplet. Verificar hashes, paridad CPU/MPS y checkpoints. Métodos externos: [DINOv2](https://arxiv.org/abs/2304.07193), [batch-hard](https://arxiv.org/abs/1703.07737). Proyección y regla de selección: decisiones de este estudio.

In [2]:
from transformers import AutoImageProcessor, AutoModel
from pettrace.evaluation.core import load_model
from pettrace.models.artifacts import Model
from pettrace.models.golden import golden_images, compare, fingerprint

model_dir = Path(snapshot_download(MODEL_REPO, revision=MODEL_REVISION,
    allow_patterns=["model.safetensors", "config.json", "preprocessor_config.json", "README.md"]))
processor = AutoImageProcessor.from_pretrained(str(model_dir), use_fast=False, trust_remote_code=False)
backbone = AutoModel.from_pretrained(str(model_dir), trust_remote_code=False).eval().requires_grad_(False)
base = load_model("models:/pettrace-embedder/2")
assert base.golden_ok and base.pipeline.detector is not None

class ReferencePipeline:
    def __init__(self, backbone, processor, detector):
        self.backbone, self.processor, self.detector = backbone, processor, detector
    def embed(self, images):
        values = []
        device = next(self.backbone.parameters()).device
        for offset in range(0, len(images), 8):
            crops = [self.detector.crop(image.convert("RGB")) for image in images[offset:offset+8]]
            pixels = self.processor(images=crops, return_tensors="pt")["pixel_values"].to(device)
            with torch.inference_mode():
                features = self.backbone(pixel_values=pixels).last_hidden_state[:, 0]
                values.append(torch.nn.functional.normalize(features, dim=-1).cpu().numpy())
        return np.concatenate(values)

pipeline = ReferencePipeline(backbone, processor, base.pipeline.detector)
golden_cpu = pipeline.embed(golden_images())
backbone.to("mps")
golden_mps = pipeline.embed(golden_images())
ok, minimum_cosine, maximum_error = compare(golden_cpu, golden_mps)
assert ok, (minimum_cosine, maximum_error)
np.save(REPORT / "golden.npy", golden_mps)
parity = {"cpu_mps_min_cosine":minimum_cosine,"cpu_mps_max_abs":maximum_error,"golden_images":12}
(REPORT / "parity.json").write_text(json.dumps(parity,indent=2))
hashes = {name:sha256(model_dir/name) for name in ("model.safetensors","config.json","preprocessor_config.json")}
hashes.update({"detector."+name:value for name,value in base.sha256.items() if "detector" in name})
assert any(name.startswith("detector.") for name in hashes)
model = Model(f"hf://{MODEL_REPO}@{MODEL_REVISION}",str(model_dir),pipeline,
              "dinov2-large-cls-yolo-crop@47b73ee", "", hashes, ok, fingerprint(golden_mps))
design = {"experiment":"C1", "method":"frozen DINOv2-L/14 CLS with E2 crop",
          "model_repo":MODEL_REPO,"model_revision":MODEL_REVISION,"model_sha256":hashes,
          "processor":processor.to_dict(),"detector_spec":base.pipeline.spec,
          "training":None,"selection":None,"test_used_for_selection":False,
          "phases":["data_preparation","representation_and_training","evaluation"],
          "gallery":"closed_set_same_species","query_unit":"photograph","capture_session_disjoint":False,
          "mlflow_publication_required":True,"register_model":False,"parity":parity,
          "notebook_method_sha256":METHOD_SHA256}
(REPORT / "experiment-design.json").write_text(json.dumps(design,indent=2))
print("Frozen model verified:",parity,"| parameters:",sum(p.numel() for p in backbone.parameters()))

from pettrace.training.dino_metric import Metric
class ProjectedPipeline:
    def __init__(self,reference,head):self.reference,self.head=reference,head
    def embed(self,images):
        x=self.reference.embed(images)
        with torch.no_grad():return self.head(torch.from_numpy(x)).numpy()
for r in recovered:
    path=Path(r['recovered'])/'metric.pt'
    assert sha256(path)==r['checkpoint_sha256']
    result=json.loads((path.parent/'result.json').read_text())
    assert abs(result['best']['mAP']-r['metrics']['val.mAP'])<1e-12
print('Three trained checkpoints verified; selected seed from validation:',selection)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/torch/nn/functional.py:4594: UserWarning: The operator 'aten::upsample_bicubic2d.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  return torch._C._nn.upsample_bicubic2d(


Frozen model verified: {'cpu_mps_min_cosine': 0.9999999403953552, 'cpu_mps_max_abs': 2.44826078414917e-05, 'golden_images': 12} | parameters: 304368640
Three trained checkpoints verified; selected seed from validation: 46


## Fase 3. Evaluación y análisis
Rank-k por identidad (máximo), mAP por fotos; recuperación cerrada de misma especie, leave-one-out por fotografía. EER descriptiva sobre pares de evaluación, no umbral operativo. Comparación pareada con C1 y bootstrap descriptivo por animal/especie con galería fija. Tres semillas; publicar los seis resultados, incluidos negativos. Limitaciones: test ya observado previamente, análisis retrospectivo, sin separación acreditada de sesiones; no implica novedad ni aceptación Q1. Los resultados se conservan en las salidas y reportes; publicación MLflow obligatoria. Registro diferido al estudio completo, sin cambiar REGISTER_MODEL=True ni promover aliases.

In [3]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.evaluation.paired import paired_comparison
publications={}
summary=[]
for r in sorted(recovered,key=lambda r:r['seed']):
    seed=r['seed']; checkpoint=Path(r['recovered'])/'metric.pt'
    head=Metric();head.load_state_dict(torch.load(checkpoint,weights_only=True,map_location='cpu'));head.eval()
    projected=ProjectedPipeline(pipeline,head)
    with torch.no_grad():expected=head(torch.from_numpy(golden_mps)).numpy()
    got=projected.embed(golden_images());assert compare(expected,got)[0]
    adapted_hashes={**hashes,'metric.pt':sha256(checkpoint)}
    adapted=Model(f"runs:/{r['run_id']}/training/metric.pt",str(checkpoint.parent),projected,
        f'dinov2-projection-seed{seed}', '', adapted_hashes,True,fingerprint(got))
    for domain,dataset in (('public',public_ds),('private',private_ds)):
        folder=REPORT/f'seed-{seed}'/domain;folder.mkdir(parents=True,exist_ok=True)
        for name in ('locked-before-test.json','dataset-selection.json','parity.json'):
            shutil.copy2(REPORT/name,folder/name)
        if (folder/'evaluation-input.json').exists():
            ev=load_publication_input(folder);ev.dataset=dataset
        else:
            started=time.monotonic();ev=evaluate(dataset,projected,zero_shot=True)
            save(ev,adapted,str(folder));save_publication_input(ev,folder)
            (folder/'runtime.json').write_text(json.dumps(dict(**SOURCE,execution_id=RUN_LABEL,
                inference_backend='torch_mps_backbone_cpu_projection',seconds=time.monotonic()-started,
                model_sha256=adapted_hashes,notebook_method_sha256=METHOD_SHA256),indent=2))
        baseline=json.loads((ROOT/'reports/c1-dinov2-reference/20261118T063806Z-cc261f74'/domain/'evaluation-input.json').read_text())
        assert baseline['dataset']['fingerprint']==dataset.fingerprint
        paired=paired_comparison(baseline['result'],ev.result)
        (folder/'paired-vs-c1.json').write_text(json.dumps(paired,indent=2))
        run=publish_evaluation(ev,adapted,folder,SOURCE['source_sha256'],experiment='D1-full-test',
            expected_model_uri=adapted.uri,expected_model_sha256=adapted_hashes,
            expected_dataset_version=dataset.version,expected_protocol='zero-shot')
        assert client.get_run(run).info.status=='FINISHED'
        receipt=dict(published=True,run_id=run,training_run=r['run_id'],seed=seed,domain=domain)
        (folder/'publication.json').write_text(json.dumps(receipt,indent=2))
        client.log_artifact(run,str(folder/'publication.json'),'report')
        downloaded=client.download_artifacts(run,'report/paired-vs-c1.json',str(folder/'verified'))
        assert sha256(downloaded)==sha256(folder/'paired-vs-c1.json')
        publications[f'{seed}-{domain}']=receipt
        summary.append(dict(seed=seed,domain=domain,metrics=ev.result['test']['global'],paired=paired,run_id=run))
        (REPORT/'summary.json').write_text(json.dumps(summary,indent=2))
        (REPORT/'publication.json').write_text(json.dumps(publications,indent=2))
        print(seed,domain,ev.result['test']['global'],'net hits',paired['net_hits'],run,flush=True)
assert len(publications)==6
print('Six test evaluations published. Full-family selection and registration remain pending.')


🏃 View run D1-full-test · dinov2-projection-seed45 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/fff1b904abcb445bae709bf5478ac609
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


45 public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9638989169675091, 'rank2': 0.9891696750902527, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.9082926790354299} net hits 12 fff1b904abcb445bae709bf5478ac609


🏃 View run D1-full-test · dinov2-projection-seed45 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/230980192ffe4a23898dfd2bdb554824
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


45 private {'queries': 502, 'gallery_animals': 33, 'rank1': 0.952191235059761, 'rank2': 0.9940239043824701, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.8924515539716765} net hits 2 230980192ffe4a23898dfd2bdb554824


🏃 View run D1-full-test · dinov2-projection-seed46 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/668bba2e3abc424bbb600cdbb7fe56c9
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


46 public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9747292418772563, 'rank2': 0.9891696750902527, 'rank5': 0.9963898916967509, 'rank10': 1.0, 'mAP': 0.909597485156355} net hits 15 668bba2e3abc424bbb600cdbb7fe56c9


... (3 outputs left out)

47 public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9638989169675091, 'rank2': 0.9855595667870036, 'rank5': 0.9963898916967509, 'rank10': 1.0, 'mAP': 0.9092098862890531} net hits 12 b7fd770e85eb4df19a1563a9ce4609f0


🏃 View run D1-full-test · dinov2-projection-seed47 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/8c2eaf8bd1e548b8ac8429856befb1cf
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


47 private {'queries': 502, 'gallery_animals': 33, 'rank1': 0.9561752988047809, 'rank2': 0.9900398406374502, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.8955515882342446} net hits 4 8c2eaf8bd1e548b8ac8429856befb1cf


Six test evaluations published. Full-family selection and registration remain pending.
